In [8]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from option import Option
from pricing_methods.black_scholes_analytic import analytic_bs_greeks
from pricing_methods.binomial import binomial_pricer
from pricing_methods.monte_carlo import mc_pricer
from pricing_methods.finite_difference import fd_pricer
from pricing_methods.pinn import pinn_pricer
from pricing_methods.greeks import all_greeks

import itertools

European

In [9]:
S0_vals = [80, 100, 120]
K = 100
sigma_vals = [0.1, 0.3, 0.6]
r_vals = [0.03, 0.07, 0.0]
div_yield_vals = [0.07, 0.03, 0.0]
T_vals = [0.1, 1.0]
type_vals = ["call", "put"]

In [10]:
grid = [Option(S0, K, T, r, sigma, div_yield, option_type, "european")
        for S0, sigma, r, div_yield, T, option_type
        in itertools.product(S0_vals, sigma_vals, r_vals, div_yield_vals,
                             T_vals, type_vals)]

In [11]:
mc_grid = [o for o in grid if o.sigma == 0.3 and o.T == 1.0]
pinn_grid = [o for o in grid if o.sigma == 0.3 and o.T == 1.0
             and o.r == 0.03 and o.option_type == "call"]

In [12]:
def grid_errors(pricer, grid, **kwargs):
    exact = [analytic_bs_greeks(o) for o in grid]
    greeks = [all_greeks(pricer, o, **kwargs) for o in grid]

    delta_errors = np.array([np.abs(exact[i][1] - greeks[i][1]) for i in range(len(exact))])
    gamma_errors = np.array([np.abs(exact[i][2] - greeks[i][2]) for i in range(len(exact))])
    vega_errors = np.array([np.abs(exact[i][3] - greeks[i][3]) for i in range(len(exact))])
    theta_errors = np.array([np.abs(exact[i][4] - greeks[i][4]) for i in range(len(exact))])
    rho_errors = np.array([np.abs(exact[i][5] - greeks[i][5]) for i in range(len(exact))])


    return (delta_errors, gamma_errors, vega_errors, theta_errors, rho_errors)

In [13]:
bin_errors = grid_errors(binomial_pricer, grid, n_steps=300)
fd_errors = grid_errors(fd_pricer, grid, n_space=300, n_steps=300)
mc_errors = grid_errors(mc_pricer, mc_grid, n_paths=200000, n_steps=100, rng=0)

In [14]:
print("Binomial Greeks Error:")
print(f"Delta: {np.mean(bin_errors[0])}")
print(f"Gamma: {np.mean(bin_errors[1])}")
print(f"Vega: {np.mean(bin_errors[2])}")
print(f"Theta: {np.mean(bin_errors[3])}")
print(f"Rho: {np.mean(bin_errors[4])}")

Binomial Greeks Error:
Delta: 0.001544991268759218
Gamma: 0.0016258529360601256
Vega: 0.10324520886873108
Theta: 0.033864231206481576
Rho: 0.008729541230120642


In [15]:
print("Monte Carlo Greeks Error:")
print(f"Delta: {np.mean(mc_errors[0])}")
print(f"Gamma: {np.mean(mc_errors[1])}")
print(f"Vega: {np.mean(mc_errors[2])}")
print(f"Theta: {np.mean(mc_errors[3])}")
print(f"Rho: {np.mean(mc_errors[4])}")

Monte Carlo Greeks Error:
Delta: 0.00035438894881182617
Gamma: 5.858111041033815e-05
Vega: 0.05041103125327385
Theta: 0.008881498203212814
Rho: 0.10237036792411401


In [16]:
print("Finite Difference Greeks Error:")
print(f"Delta: {np.mean(fd_errors[0])}")
print(f"Gamma: {np.mean(fd_errors[1])}")
print(f"Vega: {np.mean(fd_errors[2])}")
print(f"Theta: {np.mean(fd_errors[3])}")
print(f"Rho: {np.mean(fd_errors[4])}")

Finite Difference Greeks Error:
Delta: 0.0003025740191647423
Gamma: 0.0010875455826323426
Vega: 0.09072941028593359
Theta: 0.0476285150952823
Rho: 0.04778799834202271
